In [1]:
from pathlib import Path
import pandas as pd

#获取项目根目录
root = Path.cwd()

if root.name.lower() == "notebooks":
    root = root.parent

manifest_path = root / "metadata" / "manifest.csv"

df = pd.read_csv(manifest_path)

print("文件路径：",manifest_path)
print("总样本数：",len(df))

print("\n字段名称：")
print(df.columns.tolist())

print("\n各split样本数:")
print(df["split"].value_counts())

#检查speaker交集
if "speaker_id" in df.columns:
    train_speakers = set(
        df.loc[df["split"] == "train", "speaker_id"].dropna()
    )
    dev_speakers = set(
        df.loc[df["split"] == "dev", "speaker_id"].dropna()
    )
    test_speakers = set(
        df.loc[df["split"] == "test", "speaker_id"].dropna()
    )

    print("\nspeaker交叉检查：")
    print("train/dev:", len(train_speakers & dev_speakers))
    print("train/test:", len(train_speakers & test_speakers))
    print("dev/test:", len(dev_speakers & test_speakers))

#检查同一原始音频是否跨split
if "parent_uid" in df.columns:
    splits = ["train","dev","test"]
    parents = {
        s: set(df.loc[df["split"] == s,"parent_uid"].dropna())
        for s in splits
    }

    print("\nparent_uid交叉检查:")
    for a, b in [
        ("train", "dev"),
        ("train", "test"),
        ("dev", "test"),
    ]:
        print(f"{a}/{b}:", len(parents[a] & parents[b]))

print("\n是否存在session_uid字段:")
print("session_uid" in df.columns)

文件路径： d:\Projects\speech-ai-learning\metadata\manifest.csv
总样本数： 2860

字段名称：
['audio_path', 'speaker_id', 'text', 'source_id', 'parent_uid', 'start_time', 'end_time', 'split']

各split样本数:
split
train    2281
dev       409
test      170
Name: count, dtype: int64

speaker交叉检查：
train/dev: 0
train/test: 0
dev/test: 0

parent_uid交叉检查:
train/dev: 0
train/test: 0
dev/test: 0

是否存在session_uid字段:
False


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import hashlib

# 1. 项目路径
root = Path.cwd().resolve()
if root.name.lower() == "notebooks":
    root = root.parent

manifest_path = root / "metadata" / "manifest.csv"
candidate_path = root / "metadata" / "test_candidates.csv"

#2.读取完整manifest
df = pd.read_csv(manifest_path)

required = [
    "audio_path","speaker_id","text",
    "source_id","parent_uid",
    "start_time","end_time","split"
]

assert set(required).issubset(df.columns)

#3.仅使用预留test数据
test_df = df.loc[
    df["split"].astype(str).str.strip() == "test"
].copy()

print("test原始样本数：",len(test_df))

#4.计算时长，按预定5-30秒条件筛选
test_df["start_time"] = pd.to_numeric(
    test_df["start_time"], errors="coerce"
)
test_df["end_time"] = pd.to_numeric(
    test_df["end_time"], errors="coerce"
)

test_df["duration_sec"] = (
    test_df["end_time"] - test_df["start_time"]
)

eligible = test_df.loc[
    test_df["duration_sec"].between(5, 30)
].copy()

print("符合5–30秒条件的样本数:", len(eligible))

#5.检查重复记录
keys = ["audio_path","start_time","end_time"]

if eligible.duplicated(subset=keys).any():
    raise ValueError("存在重复音频片段，请先检查")

if len(eligible) < 10:
    raise ValueError("不足10条候选，需要检查时长条件")

#6.固定随即顺序
seed = 42

eligible = eligible.sort_values(
    keys,kind="stable"
).reset_index(drop=True)

rng = np.random.default_rng(seed)
eligible = eligible.iloc[
    rng.permutation(len(eligible))
].reset_index(drop=True)

eligible["selection_rank"] = np.arange(
    1, len(eligible) + 1
)

#7.生成稳定的候选UID
def make_uid(row):
    key = (
        f"{row['source_id']}|{row['audio_path']}|"
        f"{row['start_time']}|{row['end_time']}"
    )
    digest = hashlib.sha256(
        key.encode("utf-8")
    ).hexdigest()[:16]

    return f"cv26_{digest}"

eligible["candidate_uid"] = eligible.apply(
    make_uid, axis=1
)

assert eligible["candidate_uid"].is_unique

# 8. 创建人工标注字段
eligible["source_text"] = eligible["text"]

eligible["raw_text"] = ""
eligible["score_text"] = ""
eligible["status"] = "pending"
eligible["uncertain_span"] = ""
eligible["notes"] = ""
eligible["review_minutes"] = ""

eligible["review_batch"] = "backlog"
eligible.loc[
    eligible["selection_rank"] <= 10,
    "review_batch"
] = "W5D4"

eligible.loc[
    eligible["selection_rank"].between(11, 20),
    "review_batch"
] = "W5D5"

eligible["sampling_method"] = "simple_random"
eligible["sampling_seed"] = seed
eligible["session_uid"] = "unknown"

# 9. 整理字段顺序
columns = [
    "candidate_uid",
    "selection_rank",
    "review_batch",
    "source_id",
    "parent_uid",
    "speaker_id",
    "session_uid",
    "audio_path",
    "start_time",
    "end_time",
    "duration_sec",
    "split",
    "source_text",
    "raw_text",
    "score_text",
    "status",
    "uncertain_span",
    "notes",
    "review_minutes",
    "sampling_method",
    "sampling_seed"
]

candidates = eligible[columns]

# 10. 不覆盖已有标注文件
if candidate_path.exists():
    raise FileExistsError(
        f"文件已存在，请勿覆盖：{candidate_path}"
    )

candidates.to_csv(
    candidate_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n已创建:", candidate_path)
print("候选总数:", len(candidates))
print("W5D4待校对:", (
    candidates["review_batch"] == "W5D4"
).sum())
print("W5D5预留:", (
    candidates["review_batch"] == "W5D5"
).sum())

print("\nW5D4选中的10条:")
print(
    candidates.loc[
        candidates["review_batch"] == "W5D4",
        ["selection_rank", "candidate_uid", "duration_sec", "status"]
    ].to_string(index=False)
)

test原始样本数： 170
符合5–30秒条件的样本数: 98

已创建: D:\Projects\speech-ai-learning\metadata\test_candidates.csv
候选总数: 98
W5D4待校对: 10
W5D5预留: 10

W5D4选中的10条:
 selection_rank         candidate_uid  duration_sec  status
              1 cv26_cdee3e6a07874a70      6.437625 pending
              2 cv26_aed66ddddae351fa      5.489750 pending
              3 cv26_ed5ccf1debf56423      5.407500 pending
              4 cv26_6077d56f73cbebcb      9.677625 pending
              5 cv26_325f8ba312c58e72      8.285625 pending
              6 cv26_79fe04b3101be4c2      6.725625 pending
              7 cv26_76dd103842d90505      7.387500 pending
              8 cv26_93011ddb77e8016e      8.501625 pending
              9 cv26_12416afdf8547b3f      5.957625 pending
             10 cv26_7b77f50a374ed471      5.501625 pending


In [2]:
#修复manifest
from pathlib import Path
import pandas as pd
import soundfile as sf
import shutil
import numpy as np

# 1. 确定项目根目录
root = Path.cwd().resolve()
if root.name.lower() == "notebooks":
    root = root.parent

manifest_path = root / "metadata" / "manifest.csv"

audio_root = (
    root / "data" / "raw"
    / "common_voice_beijing26"
    / "zh-CN-beijing"
)

manifest = pd.read_csv(manifest_path)

# 2. 根据manifest中的路径定位真实音频
def resolve_audio_path(audio_path):
    p = Path(str(audio_path))

    if p.is_absolute():
        candidates = [p]
    else:
        candidates = [
            root / p,
            audio_root / p,
            audio_root / "clips" / p,
        ]

    for path in candidates:
        if path.is_file():
            return path

    raise FileNotFoundError(
        f"找不到音频: {audio_path}\n"
        f"尝试路径: {candidates}"
    )


# 3. 逐个读取音频时长
durations = []

for audio_path in manifest["audio_path"]:
    full_path = resolve_audio_path(audio_path)

    info = sf.info(full_path)

    duration = info.frames / info.samplerate

    if not np.isfinite(duration) or duration <= 0:
        raise ValueError(f"无效音频时长: {full_path}")

    durations.append(duration)

# 所有音频都读取成功后才修改DataFrame
manifest["start_time"] = 0.0
manifest["end_time"] = durations

# 4. 修正原先写错的数据来源版本号
if set(manifest["source_id"].dropna().unique()) == {
    "common_voice_zh-CN_17"
}:
    manifest["source_id"] = "common_voice_beijing26_zh-CN"

# 5. 打印统计，先检查
manifest["duration_sec"] = (
    manifest["end_time"] - manifest["start_time"]
)

print("音频总数:", len(manifest))

print("\n各split数量:")
print(manifest.groupby("split").size())

print("\n音频时长统计（秒）:")
print(manifest["duration_sec"].describe())

test_df = manifest[manifest["split"] == "test"]

eligible = test_df[
    test_df["duration_sec"].between(5, 30)
]

print("\ntest样本数:", len(test_df))
print("test中符合5–30秒的数量:", len(eligible))

# 6. 备份旧文件
backup_dir = root / "data" / "local_backups"
backup_dir.mkdir(parents=True, exist_ok=True)

backup_path = backup_dir / "manifest_before_end_time_fix.csv"

if backup_path.exists():
    raise FileExistsError(
        f"备份文件已存在，请先检查：{backup_path}"
    )

shutil.copy2(manifest_path, backup_path)

# 7. 保存修复后的manifest
# duration_sec只是临时统计列，保留原来的manifest字段结构
manifest = manifest.drop(columns=["duration_sec"])

temp_path = manifest_path.with_suffix(".csv.tmp")

manifest.to_csv(
    temp_path,
    index=False,
    encoding="utf-8-sig"
)

temp_path.replace(manifest_path)

print("\n修复完成:", manifest_path)
print("原文件备份:", backup_path)

音频总数: 2860

各split数量:
split
dev       409
test      170
train    2281
dtype: int64

音频时长统计（秒）:
count    2860.000000
mean        5.481321
std         1.968716
min         1.504500
25%         3.941625
50%         5.261625
75%         6.865148
max        13.147500
Name: duration_sec, dtype: float64

test样本数: 170
test中符合5–30秒的数量: 98

修复完成: D:\Projects\speech-ai-learning\metadata\manifest.csv
原文件备份: D:\Projects\speech-ai-learning\data\local_backups\manifest_before_end_time_fix.csv


In [7]:
#W5D4练习
root = Path.cwd().resolve()
if root.name.lower() == "notebook":
    root = root.parent

candidates_path = root / "metadata" / "test_candidates.csv"

candidates = pd.read_csv(candidate_path)

W5D4_sample = candidates.loc[candidates["review_batch"] == "W5D4"]
print(W5D4_sample["duration_sec"].mean())

6.937212499999999
